# Problem 1: Adversarial Search in Tic-Tac-Toe

The game engine below is fully implemented for you. It lets you place moves, check for a winner, and score a finished game with a utility from **X's perspective**:

- `+1` → X wins
- `0` → draw
- `-1` → O wins

**Your task:** write a function `estimate_move_value(move)` that determines, for a given opening move by X, the game-theoretic value of the resulting position **assuming both players play optimally from then on** (X maximizing, O minimizing). Use it to rank all 9 possible opening moves from most to least promising.

Board positions are numbered like this:

```
 0 | 1 | 2
-----------
 3 | 4 | 5
-----------
 6 | 7 | 8
```


In [ ]:
X, O, EMPTY = 'X', 'O', ' '

def new_board():
    return [EMPTY] * 9

def print_board(board):
    rows = [board[i:i+3] for i in range(0, 9, 3)]
    print('\n---------\n'.join(' | '.join(r) for r in rows))

def valid_moves(board):
    return [i for i, v in enumerate(board) if v == EMPTY]

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columns
    (0, 4, 8), (2, 4, 6)              # diagonals
]

def check_winner(board):
    """Returns 'X', 'O', 'draw', or None (game not over)."""
    for a, b, c in WIN_LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    if EMPTY not in board:
        return 'draw'
    return None

def result(board, move, player):
    """Returns a new board with `player`'s mark placed at `move`."""
    new = board.copy()
    new[move] = player
    return new

def other(player):
    return O if player == X else X

def utility(outcome):
    """Utility from X's perspective: +1 X wins, -1 O wins, 0 draw."""
    if outcome == X:
        return 1
    elif outcome == O:
        return -1
    else:
        return 0

## Quick sanity check

Run this cell to make sure the engine behaves as expected before you start the assignment.

In [ ]:
b = new_board()
b = result(b, 4, X)
b = result(b, 0, O)
print_board(b)
print('Valid moves:', valid_moves(b))
print('Winner so far:', check_winner(b))

O |   |  
---------
  | X |  
---------
  |   |  
Valid moves: [1, 2, 3, 5, 6, 7, 8]
Winner so far: None


## Your task

Implement `minimax(board, player)` and `estimate_move_value(move)` below.

- `minimax(board, player)` should return the game-theoretic utility (from X's perspective) of `board`, assuming it is `player`'s turn to move and both sides play optimally from here on: X **maximizes** the utility, O **minimizes** it.
- `estimate_move_value(move)` should start from the empty board, place X's mark at `move`, and return the value of the resulting position (it's O's turn next).

Tic-tac-toe has at most 9! ≈ 362,880 states, so plain recursion without pruning is fast enough — no need for alpha-beta here.

In [1]:
def minimax(board, player):
    """
    Returns the game-theoretic utility (from X's perspective) of `board`
    when it is `player`'s turn to move, assuming optimal play by both sides.
    """
    # TODO: implement
    # 1. If the game is already over (check_winner), return its utility.
    # 2. Otherwise, generate the value of the board that results from each
    #    valid move, recursing with the other player to move next.
    # 3. If player == X, return the max of those values.
    #    If player == O, return the min of those values.
    pass



def estimate_move_value(move):
    """
    Returns the utility (1, 0, or -1) of X opening the game with `move`,
    assuming optimal play by both sides afterward.
    """
    # TODO: implement using minimax()
    pass


## Test your implementation

This ranks all 9 opening moves by their estimated value.

In [2]:
results = {move: estimate_move_value(move) for move in valid_moves(new_board())}

for move, value in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"Move {move}: value = {value}")

## Discussion (answer)

Which opening move(s) give X the best guaranteed outcome under optimal play?

_Your answer here._

# Play Tic-Tac-Toe Against a Perfect Opponent
Try opening in the center, a corner, and an edge over a few games and see what actually happens against perfect play (spoiler: since tic-tac-toe is a solved game, every opening move draws against a perfect opponent — a corner opening isn't provably "better" in the game-theoretic sense, even though it may perform better against a *weaker* opponent).

In [3]:
def minimax(board, player):
    """Game-theoretic utility (from X's perspective) of `board` with `player` to move."""
    outcome = check_winner(board)
    if outcome is not None:
        return utility(outcome)
    values = [minimax(result(board, m, player), other(player)) for m in valid_moves(board)]
    return max(values) if player == X else min(values)

def best_ai_move(board, player):
    """Returns a move for `player` that achieves the minimax value of `board`."""
    scored = [(m, minimax(result(board, m, player), other(player))) for m in valid_moves(board)]
    if player == X:
        return max(scored, key=lambda t: t[1])[0]
    else:
        return min(scored, key=lambda t: t[1])[0]
def play_game():
    board = new_board()
    human, computer = X, O
    turn = X

    print("You are X, the computer is O. Positions are numbered 0-8, left-to-right, top-to-bottom.\n")
    print_board(board)

    while True:
        if turn == human:
            move = None
            while move not in valid_moves(board):
                raw = input("\nYour move (0-8): ")
                if raw.isdigit() and int(raw) in valid_moves(board):
                    move = int(raw)
                else:
                    print("Invalid move, try again.")
            board = result(board, move, human)
        else:
            move = best_ai_move(board, computer)
            print(f"\nComputer plays {move}")
            board = result(board, move, computer)

        print_board(board)

        outcome = check_winner(board)
        if outcome is not None:
            print()
            if outcome == 'draw':
                print("It's a draw!")
            elif outcome == human:
                print("You win! (The computer made a mistake somewhere above, or this build has a bug.)")
            else:
                print("Computer wins!")
            return

        turn = other(turn)
play_game()

# Problem 2: Min-Conflicts Search for N-Queens

In lecture we solved N-Queens with **backtracking search**: build the assignment up one variable at a time, checking consistency as you go, and undo a choice when you hit a dead end. Backtracking is a *systematic, constructive* search — it never has a complete assignment until the very end.

**Min-conflicts** is a completely different strategy: a *local search* over CSPs.

1. Start from a **complete** assignment — every variable has some value, but constraints may be violated.
2. Repeatedly pick a variable that's currently in conflict, and reassign it to whichever value minimizes its conflicts.
3. Stop as soon as the assignment has zero conflicts — that's a solution.

It sounds almost too simple to work, but for CSPs like N-Queens it's remarkably effective — it can solve boards with hundreds or thousands of queens in a fraction of a second, far beyond what plain backtracking can handle.

**Your task:** the board representation and conflict-counting helpers below are already implemented. You'll implement the **main loop** of `min_conflicts`.

In [5]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
# import matplotlib.pyplot as plt
# import matplotlib.patches as patches
# from matplotlib import animation
# from IPython.display import HTML

def random_board(n):
    """A random complete assignment: one queen per column, row chosen at random."""
    return [random.randrange(n) for _ in range(n)]

def conflicts(board, col, row):
    """
    Number of queens that would attack a queen placed at (col, row),
    given every other column's current row in `board`. Column `col`
    itself is excluded (we're asking 'what if this queen were at `row`?').
    """
    n = len(board)
    count = 0
    for c2 in range(n):
        if c2 == col:
            continue
        r2 = board[c2]
        if r2 == row or abs(r2 - row) == abs(c2 - col):
            count += 1
    return count

def total_conflicts(board):
    """Total number of attacking pairs on the board. Zero means it's a solution."""
    n = len(board)
    return sum(conflicts(board, c, board[c]) for c in range(n)) // 2

def conflicted_columns(board):
    """Columns whose queen currently conflicts with at least one other queen."""
    return [c for c in range(len(board)) if conflicts(board, c, board[c]) > 0]

def print_board(board):
    n = len(board)
    for r in range(n):
        print(' '.join('Q' if board[c] == r else '.' for c in range(n)))

def draw_board(state, N, title=''):
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    for r in range(N):
        for c in range(N):
            color = '#e8e4d8' if (r + c) % 2 == 0 else '#8a8570'
            ax.add_patch(patches.Rectangle((c, N - 1 - r), 1, 1, color=color))
    for col, row in enumerate(state):
        ax.text(col + 0.5, N - 1 - row + 0.5, '\u2655', fontsize=20, ha='center', va='center')
    ax.set_xlim(0, N)
    ax.set_ylim(0, N)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title or f'{total_conflicts(state)} conflicts')
    plt.show()

## Quick sanity check

Run this to see what a random (almost certainly conflicted) board looks like before you start.

In [6]:
random.seed(0)
b = random_board(6)
print_board(b)
print("Conflicted columns:", conflicted_columns(b))
print("Total conflicts:", total_conflicts(b))
draw_board(b,N=6)

## Your task

Implement the main loop of `min_conflicts` below, following the algorithm:

```
MIN-CONFLICTS(csp, max_steps):
    current = a random complete assignment for csp
    for i in 1 .. max_steps:
        if current is a solution: return current
        var = a randomly chosen conflicted variable
        value = the value v for var that minimizes CONFLICTS(var, v)
        set var = value in current
    return failure
```

Some things to be careful about:
- Pick the conflicted variable **at random** among conflicted columns — not always the first one — or the search can get stuck cycling.
- More than one row might tie for the minimum conflict count. Break ties **at random** too, for the same reason.
- If you run out of `max_steps` without reaching zero conflicts, return `None`.

In [7]:
def min_conflicts(n, max_steps=10000):
    """
    Runs min-conflicts local search on the N-Queens CSP.
    Returns a solved board (list of row indices, one per column),
    or None if no solution was found within max_steps.
    """
    board = random_board(n)
    # TODO: implement the main loop
    # 1. Repeat up to max_steps times:
    #    a. If total_conflicts(board) == 0, return board -- solved!
    #    b. col = a random choice from conflicted_columns(board)
    #    c. Find the row (or rows, tied) in that column that minimizes
    #       conflicts(board, col, row); break ties randomly
    #    d. Set board[col] to that row
    # 2. If the loop finishes without success, return None
    #pass

    #for _ in range(max_steps):


    return None

## Test your implementation

In [ ]:
n = 8
solution = min_conflicts(n)

if solution:
    print(f"Solved {n}-Queens:")
    print_board(solution)
    print("Total conflicts:", total_conflicts(solution))
    draw_board(solution, n)
else:
    print("No solution found within the step limit -- try increasing max_steps.")

## Push it further

Once the 8-Queens case works, try much larger boards — this is where min-conflicts really shows its strength compared to backtracking:

In [8]:
import time

for n in [50, 200, 500]:
    start = time.time()
    solution = min_conflicts(n, max_steps=100000)
    elapsed = time.time() - start
    status = "solved" if solution and total_conflicts(solution) == 0 else "failed"
    print(f"n={n}: {status} in {elapsed:.3f}s")

## Discussion (answer in 1-2 sentences)

Can you think of a type of CSP where you'd expect min-conflicts to struggle instead?

_Your answer here._